## Dependencies

In [320]:
## libraries
import sys
import logging
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.estimators.factories import load_estimators
from src.data.builders import (
    load_processed_data, 
    load_falsified_data
)
from src.evaluators.falsifying import (
    eval_falsified_frontier,
    eval_falsified_alignment,
    eval_falsified_consensus,
    stat_falsified_test,
    stat_falsified_summary,
)
from src.evaluators.metrics import (
    FRONTIER_METRICS,
    CONSENSUS_METRICS
)
from src.evaluators.config import (
    FEAT_X, 
    FEAT_Z, 
    TARGET
)

## Initialization

In [321]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data_proc = load_processed_data()
    data_fals = load_falsified_data()
finally:
    logging.disable(_disable)
    models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data_proc.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")
print("Falsified Data:")
for method, data in data_fals.items():
    n_r, n_c = data.shape
    print(f" - {method}: {n_c} features, {n_r} observations")

Original Data: 32 features, 25 observations
Falsified Data:
 - random_generate: 32 features, 25 observations
 - target_remap: 32 features, 25 observations
 - vector_generate: 32 features, 25 observations


## Frontier Falsification Evaluation

Three falsification types (target remap, random generate, vector generate) are applied to the processed datasets while the evaluation target remains fixed. Model sensitivity is evaluated with LOGO-CV (domain) under frozen and retrained protocols, allowing original-data frontier performance to be compared against deliberately corrupted inputs.

In [322]:
## test falsified frontier
if "results_falsified_frontier" not in globals():
    results_falsified_frontier = eval_falsified_frontier(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        group = 'domain',
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE
    )

## Frontier Falsifiability Test

This section tests whether models produce stronger frontier envelope metrics on original data than on falsified data. Under the frozen protocol, models trained on original data are evaluated directly on falsified inputs. Under the retrain protocol, models are refit on falsified data and compared back to the original-data baseline.

In [323]:
## wilcoxon signed-rank test for falsified frontier
results_table_frontier = stat_falsified_test(
    results = results_falsified_frontier,
    feat_value = ["ei"],
    feat_pairs = ["model", "group"]
 )

display(results_table_frontier)

Paired One-Sided Test (Wilcoxon Signed-Rank): n = 45
H₀: Δ EI ≤ 0
H₁: Δ EI > 0
Median Δ EI: Median of paired differences, not the difference of marginal medians
Rank-biserial r: Paired effect size, positive values favor original
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Diff.: Yes if Holm-adj. p < 0.05 and Median Δ > 0
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


Median Δ EI Rank-biserial r One-sided p Holm-adj. p  \
Frontier Method                                                                
frozen   target remap         0.0291          0.3739      0.0142      0.0142   
         random generate      0.0532          0.8415      0.0000      0.0000   
         vector generate      0.0229          0.6638      0.0000      0.0001   
retrain  target remap         0.0631          0.5420      0.0006      0.0021   
         random generate      0.0474          0.5478      0.0005      0.0021   
         vector generate      0.0218          0.5188      0.0010      0.0021   

                         Sig. Diff.  
Frontier Method                      
frozen   target remap       *   Yes  
         random generate  ***   Yes  
         vector generate  ***   Yes  
retrain  target remap      **   Yes  
         random generate   **   Yes  
         vector generate   **   Yes

In [324]:
## median falsified frontier metrics
frontier_summary = stat_falsified_summary(
    results = results_falsified_frontier,
    metrics = FRONTIER_METRICS
)

display(frontier_summary)

vr      mv      ms      ei
Frontier Method                                      
original original         0.0  0.0000  5.1100  0.7994
frozen   target remap     0.2  0.4003  5.0082  0.7612
         random generate  0.0  0.0000  7.9842  0.7466
         vector generate  0.0  0.0000  6.8557  0.7511
retrain  target remap     0.0  0.0000  5.8559  0.7451
         random generate  0.0  0.0000  5.7517  0.7468
         vector generate  0.0  0.0000  5.2695  0.7527

All three falsification types yield significant EI degradation under both the frozen and retrained protocols, with effect sizes comparable across types. Target remap corrupts the target only, while random and vector generation corrupt the input features, yet the resulting EI losses are of similar magnitude. Retraining does not restore original EI performance for any falsification type, and for target remap the retrained contrast is the largest observed shift.

## Target-Alignment Evaluation

The same three falsification types are applied while the observed target remains unchanged, isolating whether predictive alignment depends on coherent graph-derived structure rather than on incidental feature variation. Model performance is evaluated with LOGO-CV (domain) under frozen and retrained protocols using cross-validated predictions on the falsified inputs.

In [325]:
## test falsified target alignment
if "results_falsified_alignment" not in globals():
    results_falsified_alignment = eval_falsified_alignment(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        group = "domain",
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE
    )

## Target-Alignment Test

This section tests whether original data yields stronger agreement between model predictions and the observed target than falsified data. It evaluates global rank and correlation alignment under the frozen and retrained protocols, treating each falsification method as an independent destructive intervention on the input signal.

In [326]:
## wilcoxon signed-rank test for falsified target alignment
results_table_alignment = stat_falsified_test(
    results = results_falsified_alignment,
    feat_value = ["ci"],
    feat_pairs = ["model", "group"]
)

display(results_table_alignment)

Paired One-Sided Test (Wilcoxon Signed-Rank): n = 9
H₀: Δ CI ≤ 0
H₁: Δ CI > 0
Median Δ CI: Median of paired differences, not the difference of marginal medians
Rank-biserial r: Paired effect size, positive values favor original
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Diff.: Yes if Holm-adj. p < 0.05 and Median Δ > 0
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


Median Δ CI Rank-biserial r One-sided p Holm-adj. p  \
Frontier Method                                                                
frozen   target remap         0.5790          1.0000      0.0020      0.0117   
         random generate      0.5341          1.0000      0.0020      0.0117   
         vector generate      0.3484          0.9556      0.0039      0.0117   
retrain  target remap         0.4754          1.0000      0.0020      0.0117   
         random generate      0.5341          1.0000      0.0020      0.0117   
         vector generate      0.5341          1.0000      0.0020      0.0117   

                         Sig. Diff.  
Frontier Method                      
frozen   target remap       *   Yes  
         random generate    *   Yes  
         vector generate    *   Yes  
retrain  target remap       *   Yes  
         random generate    *   Yes  
         vector generate    *   Yes

In [327]:
## median falsified target-alignment metrics
alignment_summary = stat_falsified_summary(
    results = results_falsified_alignment,
    metrics = CONSENSUS_METRICS
)

display(alignment_summary)

rho     rbo     dcr      ci
Frontier Method                                         
original original         0.5623  0.5738  0.5812  0.5791
frozen   target remap    -0.1908  0.2908  0.3300  0.0000
         random generate -0.1638  0.3120  0.3973  0.0001
         vector generate  0.0900  0.3782  0.2925  0.2138
retrain  target remap     0.0146  0.3046  0.2903  0.1118
         random generate -0.1354  0.3390  0.3195  0.0000
         vector generate -0.2715  0.3224  0.4297  0.0001

Every falsification type produces a significant loss of target alignment under both protocols, with rank-biserial effect sizes at or near the theoretical maximum. Rank correlation with the observed target collapses from positive under the original baseline to near zero or negative under every falsified setting, including target remap. Retraining does not restore alignment; several retrained variants remain as far below the original as their frozen counterparts.

## Pairwise Consensus Evaluation

The same falsification types are applied to assess whether fitted frontier structure remains stable when informative input geometry is destroyed. Pairwise agreement is evaluated across model frontiers under frozen and retrained protocols, focusing on inter-model consistency rather than direct prediction of the observed target.

In [328]:
## test falsified pairwise consensus
if "results_falsified_consensus" not in globals():
    results_falsified_consensus = eval_falsified_consensus(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE
    )

## Pairwise Consensus Test

This section compares fitted model frontiers to one another rather than comparing each model to the observed target. It tests whether inter-model agreement declines under falsification for the frozen and retrained protocols, framing pairwise consensus as a fitted-frontier stability analysis rather than a predictive resampling test.

In [329]:
## wilcoxon signed-rank test for falsified pairwise consensus
results_table_pairwise = stat_falsified_test(
    results = results_falsified_consensus,
    feat_value = ["ci"],
    feat_pairs = ["model_i", "model_j", "group"]
)

display(results_table_pairwise)

Paired One-Sided Test (Wilcoxon Signed-Rank): n = 36
H₀: Δ CI ≤ 0
H₁: Δ CI > 0
Median Δ CI: Median of paired differences, not the difference of marginal medians
Rank-biserial r: Paired effect size, positive values favor original
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Diff.: Yes if Holm-adj. p < 0.05 and Median Δ > 0
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


Median Δ CI Rank-biserial r One-sided p Holm-adj. p  \
Frontier Method                                                                
frozen   target remap         0.0000          0.8000      0.0721      0.0721   
         random generate      0.7660          0.9910      0.0000      0.0000   
         vector generate      0.5025          0.9970      0.0000      0.0000   
retrain  target remap         0.4905          1.0000      0.0000      0.0000   
         random generate      0.3030          0.9009      0.0000      0.0000   
         vector generate      0.1730          0.7387      0.0000      0.0000   

                         Sig. Diff.  
Frontier Method                      
frozen   target remap            No  
         random generate  ***   Yes  
         vector generate  ***   Yes  
retrain  target remap     ***   Yes  
         random generate  ***   Yes  
         vector generate  ***   Yes

In [330]:
## median pairwise consensus metrics
pairwise_summary = stat_falsified_summary(
    results = results_falsified_consensus,
    metrics = CONSENSUS_METRICS
)

display(pairwise_summary)

rho     rbo     dcr      ci
Frontier Method                                         
original original         0.9596  0.8665  0.9506  0.9193
frozen   target remap     0.9596  0.8665  0.9506  0.9193
         random generate  0.0565  0.4012  0.3016  0.1731
         vector generate  0.3478  0.5302  0.4083  0.4134
retrain  target remap     0.3235  0.5328  0.4639  0.3897
         random generate  0.5200  0.5399  0.5988  0.5049
         vector generate  0.7216  0.7737  0.7300  0.7360

Frozen target remap is identical to the original baseline because it leaves the input features untouched and changes only the target, so fitted frontiers are unchanged by construction and the paired test is undefined. Every other contrast shows a significant decline in pairwise agreement, with the sharpest collapse under frozen random generation. Retraining partially recovers consensus for random and vector generation, but for target remap it degrades consensus further by refitting frontiers to the remapped target.